# 05 · Learnable Aggregators + the full 4-way RQ1 study

`mean` and `centroid` are fixed heuristics. This trains the two **learnable** aggregators — **Deep-Sets** and **attention-PMA** — then re-runs RQ1 as a **4-way** comparison against the retrained (`diffusion_v2`) model.

**Training objective (self-supervised, no diffusion in the loop):** given a noisy crowd's CLIP embeddings, produce a vector close to the crowd's *clean theme* embedding — i.e. learn robust theme extraction. Fast (~2–3 min).

> Runtime → **GPU (T4 is fine)** — aggregators are tiny; the eval is a few hundred small samples.

## 1. Clone & install

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation
!pip install -q open-clip-torch matplotlib
import torch; print('cuda', torch.cuda.is_available())

## 2. Credentials & locate checkpoints
Needs `vae.pt` and the **retrained** `diffusion_v2/diffusion.pt` (Drive first, else HF).

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')
hf_token = getpass.getpass('HF token (Enter to skip if on Drive): ').strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token)

def locate(drive_path, repo, fname):
    if os.path.exists(drive_path):
        return drive_path
    from huggingface_hub import hf_hub_download, whoami
    return hf_hub_download(f"{whoami()['name']}/{repo}", fname)

VAE_CKPT  = locate('/content/drive/MyDrive/crowdgen/vae/vae.pt', 'crowdgen-vae', 'vae.pt')
DIFF_CKPT = locate('/content/drive/MyDrive/crowdgen/diffusion_v2/diffusion.pt', 'crowdgen-diffusion-v2', 'diffusion.pt')
OUT = '/content/drive/MyDrive/crowdgen/diffusion_v2'
AGG = f'{OUT}/aggregators.pt'
print('VAE :', VAE_CKPT); print('DIFF:', DIFF_CKPT)

## 3. Train the learnable aggregators
Deep-Sets + attention-PMA, ~1500 steps. Watch `cos-loss` fall — final cos-sim to the clean theme near ~0.5+ means they're learning robust extraction. Trained at `--n 300` to match evaluation.

In [ ]:
!python train_aggregators.py --diffusion {DIFF_CKPT} \
    --aggregators deepsets,attention --n 300 --batch 16 --steps 1500 --out {AGG}

## 4. The full 4-way RQ1 study
`mean` · `centroid` · `deepsets` · `attention`, scored on the retrained model. Do the *learned* aggregators beat the heuristics at holding theme-fidelity as crowd noise rises? ~5–8 min.

In [ ]:
!python evaluate.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} --agg-ckpt {AGG} \
    --aggregators mean,centroid,deepsets,attention \
    --repeats 8 --n 300 --guidance 3.0 --steps 50 --out {OUT}/rq1_4way
from IPython.display import Image, display
display(Image(f'{OUT}/rq1_4way/rq1_fidelity.png')); display(Image(f'{OUT}/rq1_4way/rq1_consistency.png'))

## Next
This completes RQ1 as a 4-way study. Compare the four lines (in `FINDINGS.md`): typically expect the **learned** aggregators (attention/Deep-Sets) ≥ **centroid** > **mean**, especially at high diversity. Remaining: bump `--repeats` + error bars for rigor, optionally regenerate the qualitative crowd→image grids (`03`) on `diffusion_v2`, then write the report from `FINDINGS.md`.